# baseline

이 베이스라인 코드는 `사전학습 모델 로드`, `배치 학습`, `파인튜닝`, `양자화`, `PEFT` 등이 적용된 버전입니다.

윈도우 데스크탑의 RTX 5060 ti GPU 환경에서 개발되었습니다.

# 환경 준비

개발 환경에 필요한 라이브러리 버전을 고정하고 최신 버전으로 라이브러리를 업데이트합니다.

- 아래 셀 실행
- ipykernel 설치
- 아래 셀 다시 실행 : 무한 로딩 시 restart
- hello 출력시 torch 설치

In [1]:
print('hello123')

hello123


In [2]:
!pip install --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Looking in indexes: https://download.pytorch.org/whl/nightly/cu128



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import torch

print(torch.__version__)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name())

2.11.0+cu128
True
NVIDIA GeForce RTX 5060 Ti


In [4]:
# !pip -q install "transformers>=4.43.2,<5.0.0" "accelerate>=0.34.2" "peft>=0.13.2" "bitsandbytes>=0.43.3" datasets pillow pandas --upgrade 
!pip -q install -U git+https://github.com/huggingface/transformers accelerate peft bitsandbytes datasets pillow pandas


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 데이터 준비

개발에 필요한 데이터를 준비합니다.

- train.csv, train 폴더
- test.csv, test 폴더
- sample_submission.csv

데이터를 압축 해제하는데 몇 분 정도의 시간이 소요됩니다.

# 라이브러리, 데이터, 설정

In [5]:
import transformers
print(transformers.__version__)

d:\docker_image\2026-ssafy-15-2-ai\baseline\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


5.5.0.dev0


In [6]:
# ===== 기본 환경 설정 및 데이터 로드 =====
import transformers
print(transformers.__version__)

import os, math, random
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
import torch
from typing import Any
from transformers import (
    AutoModelForImageTextToText,
    AutoProcessor,
    BitsAndBytesConfig,
    get_linear_schedule_with_warmup
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from tqdm import tqdm

Image.MAX_IMAGE_PIXELS = None

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

MODEL_ID = "Qwen/Qwen3-VL-8B-Instruct"
IMAGE_SIZE = 384
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DATA_ROOT = r"D:\docker_image\2026-ssafy-15-2-ai"

DEBUG = False
DEBUG_N = 300

USE_VALID = False
VAL_RATIO = 0.1
USE_DEV_PSEUDO = False

train_df = pd.read_csv(os.path.join(DATA_ROOT, "train.csv"))
test_df  = pd.read_csv(os.path.join(DATA_ROOT, "test.csv"))
dev_df   = pd.read_csv(os.path.join(DATA_ROOT, "dev.csv"))

if DEBUG:
    train_df = train_df.sample(n=min(DEBUG_N, len(train_df)), random_state=SEED).reset_index(drop=True)

5.5.0.dev0
Device: cuda


# 모델, Processor

7.5GB 정도의 모델 다운로드가 진행됩니다. 10~20분 정도가 소요됩니다.

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - LoRA 구현 : LoraConfig()

In [7]:
# ===== dev 데이터를 다수결 기반 pseudo label로 활용 (선택 옵션) =====
def maybe_get_dev_pseudo(df):
    candidate_cols = [c for c in df.columns if "응답" in c]

    out_rows = []
    for _, row in df.iterrows():
        votes = []
        for c in candidate_cols:
            v = str(row[c]).strip().lower()
            if v in ["a", "b", "c", "d"]:
                votes.append(v)

        if not votes:
            continue

        cnt = pd.Series(votes).value_counts()
        if cnt.iloc[0] >= 3:
            out_rows.append({
                "id": row["id"],
                "path": row["path"],
                "question": row["question"],
                "a": row["a"],
                "b": row["b"],
                "c": row["c"],
                "d": row["d"],
                "answer": cnt.index[0]
            })

    return pd.DataFrame(out_rows)


if USE_DEV_PSEUDO:
    pseudo = maybe_get_dev_pseudo(dev_df)
    train_df = pd.concat([train_df, pseudo], ignore_index=True)

In [8]:
# ===== 4bit 양자화 + LoRA 기반 VLM 모델 로드 =====
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = "left"

base_model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)

W0402 20:18:19.546000 19260 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Loading weights:   0%|          | 2/750 [00:04<33:55,  2.72s/it]d:\docker_image\2026-ssafy-15-2-ai\baseline\Lib\site-packages\bitsandbytes\backends\cuda\ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
Loading weights: 100%|██████████| 750/750 [00:53<00:00, 14.12it/s] 


# 프롬프트 템플릿

#### 실습 참고 내용

    챕터 5-1 PEFT(파라미터 효율적 튜닝)
    - 프롬프트 템플릿 : convert_to_chatml(), formatting_prompts_func()

In [9]:
# ===== 문제를 단순한 4지선다 형태로 입력하는 프롬프트 =====
SYSTEM_INSTRUCT = "Choose correct answer: a, b, c, or d."

CHOICE_KEYS = ["a","b","c","d"]

def build_prompt(row):
    return (
        f"{row['question']}\n"
        f"(a) {row['a']}\n"
        f"(b) {row['b']}\n"
        f"(c) {row['c']}\n"
        f"(d) {row['d']}\n\n"
        f"Answer:"
    )

# Custom Dataset, Collator

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - TensorDataset()

    챕터 5-2 데이터 생성 및 파인튜닝 (향후 학습 분량)
    - IntentDataset()

In [10]:
# ===== 이미지 + 텍스트를 묶어 모델 입력 형태로 변환 =====
def resolve_path(path):
    return path if os.path.isabs(path) else os.path.join(DATA_ROOT, path)

class VQADataset(Dataset):
    def __init__(self, df, mode="train"):
        self.df = df.reset_index(drop=True)
        self.mode = mode

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]
        img = Image.open(resolve_path(row["path"])).convert("RGB")

        prompt = build_prompt(row)

        messages = [
            {"role":"system","content":[{"type":"text","text":SYSTEM_INSTRUCT}]},
            {"role":"user","content":[
                {"type":"image","image":img},
                {"type":"text","text":prompt}
            ]}
        ]

        item = {"messages":messages, "image":img}

        if self.mode=="train":
            item["gold"] = row["answer"]

        return item

In [11]:
# ===== 정답 토큰(a/b/c/d) 위치만 학습하도록 loss 마스킹 =====
@dataclass
class AnswerOnlyCollator:
    processor: Any
    train: bool = True

    def __call__(self, batch):
        prompt_texts, full_texts, images = [], [], []

        for s in batch:
            prompt = self.processor.apply_chat_template(
                s["messages"],
                tokenize=False,
                add_generation_prompt=True
            )
            prompt_texts.append(prompt)
            images.append(s["image"])

            if self.train:
                full_texts.append(prompt + str(s["gold"]).strip().lower())

        enc = self.processor(
            text=full_texts if self.train else prompt_texts,
            images=images,
            padding=True,
            return_tensors="pt"
        )

        if not self.train:
            return enc

        labels = torch.full_like(enc["input_ids"], -100)
        seq_len = enc["input_ids"].shape[1]

        for i in range(len(batch)):
            valid_len = int(enc["attention_mask"][i].sum().item())
            start_idx = seq_len - valid_len
            answer_pos = seq_len - 1   # 마지막 토큰이 gold 한 글자
            labels[i, answer_pos] = enc["input_ids"][i, answer_pos]

        enc["labels"] = labels
        return enc

# DataLoader

#### 실습 참고 내용

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 데이터로더 정의 : DataLoader()

In [12]:
# ===== 학습용 데이터 로더 구성 =====
train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

train_ds = VQADataset(train_df, "train")

train_loader = DataLoader(
    train_ds,
    batch_size=1,
    shuffle=True,
    collate_fn=AnswerOnlyCollator(processor, True)
)

# fine-tuning

- 200개만 학습 : 10~20분 소요

#### 실습 참고 내용

    챕터 1-2 MLP 구현
    - 모델 정의 : SimpleMLP(), SequentialMLP()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [13]:
# ===== LoRA 파인튜닝 학습 루프 =====
# model = model.to(device)

GRAD_ACCUM = 4
EPOCHS = 1
LR = 1e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
num_training_steps = EPOCHS * math.ceil(len(train_loader) / GRAD_ACCUM)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    int(num_training_steps * 0.03),
    num_training_steps
)

scaler = torch.amp.GradScaler("cuda", enabled=torch.cuda.is_available())

for epoch in range(EPOCHS):
    model.train()
    running = 0.0
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1} [train]", unit="batch")

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(progress_bar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            out = model(**batch)
            loss = out.loss / GRAD_ACCUM

        scaler.scale(loss).backward()
        running += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            avg_loss = running / min(GRAD_ACCUM, step)
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}"})
            running = 0.0

Epoch 1 [train]:   0%|          | 0/5073 [00:00<?, ?batch/s]d:\docker_image\2026-ssafy-15-2-ai\baseline\Lib\site-packages\torch\_dynamo\eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
d:\docker_image\2026-ssafy-15-2-ai\baseline\Lib\site-packages\torch\utils\checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.
Epoch 1 [train]:   0%|          | 3/5073 [00:05<2:12:49,  1.57s/batch]C:\Users\SSAFY\AppData\Local\Temp\ipykernel_19260\3286894398.py

# inference

30분~1시간 소요

#### 실습 참고 내용

    챕터4-1 RAG 기반 Customer Service AI 에이전트 개발
    - 데이터 파서 : langchain_core.output_parsers(), StrOutputParser()

    챕터 3-1 Transfer Learning 기반의 CNN 모델 학습
    - 학습 루프 : 문제 6: 모델 학습을 위한 반복문
    - 추론 : with torch.no_grad(), model.eval()

In [14]:
# ===== generate 대신 a/b/c/d logits 비교로 직접 선택 =====
choice_ids = {}
for c in CHOICE_KEYS:
    ids = processor.tokenizer.encode(c, add_special_tokens=False)
    if len(ids) != 1:
        raise ValueError(f"{c} is not a single token: {ids}")
    choice_ids[c] = ids[0]

model.eval()
preds = []

test_ds = VQADataset(test_df, "test")
test_loader = DataLoader(
    test_ds,
    batch_size=4,
    shuffle=False,
    collate_fn=AnswerOnlyCollator(processor, False)
)

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Inference", unit="batch"):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            out = model(**batch)

        logits = out.logits
        row_idx = torch.arange(logits.size(0), device=device)

        # left padding이면 마지막 입력 토큰은 항상 맨 끝
        last_pos = torch.full(
            (logits.size(0),),
            logits.size(1) - 1,
            dtype=torch.long,
            device=device
        )

        next_logits = logits[row_idx, last_pos, :]

        choice_logits = torch.stack(
            [next_logits[:, choice_ids[c]] for c in CHOICE_KEYS],
            dim=1
        )

        pred_idx = choice_logits.argmax(dim=1).tolist()
        preds.extend([CHOICE_KEYS[i] for i in pred_idx])

Inference: 100%|██████████| 1269/1269 [20:55<00:00,  1.01batch/s]


In [15]:
# 제출 파일 생성
submission = pd.DataFrame({"id": test_df["id"], "answer": preds})
submission.to_csv("./outputs/submission.csv", index=False)
print("Saved outputs/submission.csv")

Saved outputs/submission.csv


In [16]:
# 모델 응답 예시
print(preds[:10])

['d', 'a', 'c', 'a', 'b', 'b', 'a', 'b', 'd', 'b']
